# KOSIS BGE-M3 hybrid retrieval on Colab Pro+

107,138개 KOSIS 통계표의 dense index를 Google Drive에 만들고, BGE multilingual reranker까지 확인합니다. 런타임이 끊겨도 같은 출력 폴더로 다시 실행하면 `progress.json`의 완료 행부터 재개합니다.

Colab 메뉴에서 **런타임 > 런타임 유형 변경 > GPU**를 먼저 선택하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/rnwjdgus03/NLP_05-Team-Project-3.git'
BRANCH = 'Poc'
REPO_DIR = Path('/content/NLP_05-Team-Project-3')
DRIVE_ROOT = Path('/content/drive/MyDrive/NLP_05-Team-Project-3')
INDEX_DIR = DRIVE_ROOT / 'indexes' / 'kosis_bge_m3'
HANDOFF_CSV = DRIVE_ROOT / 'inputs' / 'hcx_extracted_handoff_100_v15.csv'
RUN_DIR = DRIVE_ROOT / 'runs' / 'kosis_hybrid_colab'
BATCH_SIZE = 64

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)
print('index:', INDEX_DIR)

In [ ]:
import os
import subprocess

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
requirements = REPO_DIR / 'requirements-ml.txt'
builder = REPO_DIR / 'kosis_build_embedding_index.py'
if not requirements.exists() or not builder.exists():
    raise FileNotFoundError(
        f'{BRANCH} 원격 브랜치에 임베딩 코드가 없습니다. 최신 커밋을 push한 뒤 런타임을 다시 실행하세요.'
    )
subprocess.run(['python', '-m', 'pip', 'install', '-r', str(requirements)], check=True)
print('repo:', REPO_DIR)

## 전체 통계표 임베딩 인덱스 생성

기본 모델은 `BAAI/bge-m3`입니다. 완료 산출물은 `embeddings.npy`, `tables.csv`, `manifest.json`입니다. 세션 중단 시 이 셀을 다시 실행하면 마지막 체크포인트부터 이어집니다. 완전히 새로 만들 때만 명령에 `--force`를 추가하세요.

In [ ]:
import subprocess
import sys

TABLE_INDEX = REPO_DIR / 'kosis_table_summary.csv'
command = [
    sys.executable,
    str(REPO_DIR / 'kosis_build_embedding_index.py'),
    '--table-index', str(TABLE_INDEX),
    '--out-dir', str(INDEX_DIR),
    '--embedding-model', 'BAAI/bge-m3',
    '--batch-size', str(BATCH_SIZE),
    '--device', 'cuda',
]
subprocess.run(command, check=True)

In [ ]:
import json

manifest = json.loads((INDEX_DIR / 'manifest.json').read_text(encoding='utf-8'))
print(json.dumps(manifest, ensure_ascii=False, indent=2))
for path in sorted(INDEX_DIR.iterdir()):
    print(path.name, f'{path.stat().st_size / 1024 / 1024:.1f} MiB')

## Reranker 단독 확인

인덱스 생성 프로세스가 끝난 뒤 실행하므로 GPU 메모리가 정리된 상태에서 `BAAI/bge-reranker-v2-m3`를 확인합니다. 첫 문서가 두 번째 문서보다 높은 점수를 받아야 정상입니다.

In [ ]:
from kosis_semantic_search import TransformerReranker

reranker = TransformerReranker('BAAI/bge-reranker-v2-m3', device='cuda', batch_size=4)
query = '지표: 반도체 수출액 | 대상: 반도체 | 단위: 달러 | 주기: 연간'
documents = [
    '통계표: 품목별 수출액, 수입액 | 분류경로: 무역통계',
    '통계표: 맞벌이 가구 비율 | 분류경로: 가구통계',
]
scores = reranker.score(query, documents)
print(scores)
assert scores[0] > scores[1]

## 100행 hybrid 후보 검색

Google Drive의 `NLP_05-Team-Project-3/inputs/`에 `hcx_extracted_handoff_100_v15.csv`를 넣은 뒤 실행하세요. 먼저 `--skip-meta`로 임베딩 검색과 리랭킹 결과만 확인합니다. 후보 CSV의 `retrieval_backend`, `semantic_score`, `reranker_score`, `fusion_score`가 채워져야 합니다.

In [ ]:
import subprocess
import sys

if not HANDOFF_CSV.exists():
    raise FileNotFoundError(f'인계 CSV를 여기에 넣으세요: {HANDOFF_CSV}')
RUN_DIR.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable,
    str(REPO_DIR / 'run_kosis_measurement_pipeline.py'),
    '--input', str(HANDOFF_CSV),
    '--table-index', str(TABLE_INDEX),
    '--out-dir', str(RUN_DIR),
    '--retrieval-mode', 'hybrid',
    '--semantic-index', str(INDEX_DIR),
    '--semantic-top-k', '50',
    '--rerank-top-k', '20',
    '--device', 'cuda',
    '--skip-meta',
]
subprocess.run(command, check=True)

In [ ]:
import pandas as pd

candidate_path = RUN_DIR / 'hcx_extracted_handoff_100_v15_kosis_table_candidates.csv'
candidates = pd.read_csv(candidate_path, encoding='utf-8-sig')
display(candidates[[
    'claim_measurement_id', 'indicator', 'candidate_rank', 'tbl_name',
    'retrieval_backend', 'lexical_score', 'lexical_eligible', 'semantic_score',
    'reranker_score', 'fusion_score'
]].head(30))
print(candidates['retrieval_backend'].value_counts(dropna=False))
top1 = candidates[candidates['candidate_rank'] == 1]
print('top1 measurements:', len(top1))
print('top1 lexical eligible:', top1['lexical_eligible'].value_counts(dropna=False).to_dict())
display(top1[top1['lexical_eligible'] != 'Y'][[
    'claim_measurement_id', 'indicator', 'tbl_name', 'lexical_score',
    'semantic_score', 'reranker_score', 'fusion_score'
]])

## KOSIS 메타 조회와 최종 매핑 상태

Colab 왼쪽의 **보안 비밀(열쇠 아이콘)**에 `KOSIS_API_KEY`를 등록하세요. 검토를 마친 기존 후보 CSV를 재사용하므로 임베딩과 리랭커는 다시 실행하지 않습니다. 이 단계는 실제값 검증 전 `READY/REVIEW/REJECT`까지만 생성합니다.

In [ ]:
import os
from google.colab import userdata

if not os.environ.get('KOSIS_API_KEY'):
    try:
        os.environ['KOSIS_API_KEY'] = userdata.get('KOSIS_API_KEY')
    except Exception as exc:
        raise RuntimeError(
            'Colab 보안 비밀에 KOSIS_API_KEY를 등록하고 노트북 액세스를 허용하세요.'
        ) from exc
if not os.environ.get('KOSIS_API_KEY'):
    raise RuntimeError('KOSIS_API_KEY가 비어 있습니다.')
print('KOSIS API key: loaded')

In [ ]:
meta_command = [
    sys.executable,
    str(REPO_DIR / 'run_kosis_measurement_pipeline.py'),
    '--input', str(HANDOFF_CSV),
    '--table-index', str(TABLE_INDEX),
    '--out-dir', str(RUN_DIR),
    '--reuse-table-candidates',
    '--top-rank-for-meta', '2',
    '--top-meta', '8',
    '--delay', '0.15',
]
subprocess.run(meta_command, check=True)

In [ ]:
final_path = RUN_DIR / 'hcx_extracted_handoff_100_v15_kosis_candidates_with_meta.csv'
mapped = pd.read_csv(final_path, encoding='utf-8-sig')
mapped_top1 = mapped[mapped['candidate_rank'] == 1].copy()
print('top1 status:', mapped_top1['candidate_status'].value_counts(dropna=False).to_dict())
display(mapped_top1[[
    'claim_measurement_id', 'indicator', 'tbl_name', 'candidate_status',
    'candidate_status_code', 'candidate_status_reason',
    'selected_itm_name', 'selected_itm_unit', 'selected_obj_l1_name',
    'mapping_type', 'unit_compatibility_reason'
]])
print('final:', final_path)